# Register the tree ensemble (XGBoost + LightGBM)

Wraps the clean blend as an MLflow **pyfunc** so it can be loaded from the registry and
run on the **raw** `test.csv`, exactly like the TimesFM champion. The wrapper bundles
both fitted pipelines, the blend weight, and the raw train/stores/features so it can
rebuild the test features (with continuous deltas) at inference time.

Registered as `Walmart_Tree_Ensemble`. Note: TimesFM is still the overall champion; this
is the best tree-only model, registered for completeness and clean loading.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import warnings
import numpy as np
import pandas as pd
from sklearn.base import clone

from src.features.preprocessing import get_model_ready_data
from src.models.xgboost_pipeline import create_xgboost_pipeline
from src.models.lightgbm_pipeline import create_lightgbm_pipeline

warnings.filterwarnings('ignore')

# Final chosen configs and blend weight (from model_ensemble_trees.ipynb).
XGB_PARAMS = {'objective': 'reg:squarederror', 'n_estimators': 300, 'max_depth': 12,
              'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.9,
              'random_state': 42, 'n_jobs': -1}
LGB_PARAMS = {'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'subsample_freq': 1,
              'n_estimators': 2000, 'learning_rate': 0.02, 'num_leaves': 511,
              'min_child_samples': 5, 'subsample': 1.0, 'colsample_bytree': 0.9, 'max_depth': -1}
W_XGB = 0.45

train_raw = pd.read_csv('data/train.csv')
stores = pd.read_csv('data/stores.csv')
features = pd.read_csv('data/features.csv')

# Fit both models on ALL train data (train + val).
X_train, y_train, X_val, y_val, _, preprocessor = get_model_ready_data(
    train_raw, stores, features, split_date='2012-01-01'
)
X_all = pd.concat([X_train, X_val], axis=0)
y_all = pd.concat([y_train, y_val], axis=0)

xgb_final = create_xgboost_pipeline(clone(preprocessor), XGB_PARAMS)
xgb_final.fit(X_all, y_all)
lgb_final = create_lightgbm_pipeline(clone(preprocessor), LGB_PARAMS)
lgb_final.fit(X_all, y_all)
print('fitted both models on', X_all.shape[0], 'rows')

fitted both models on 421570 rows


## The ensemble pyfunc (runs on raw test.csv)

In [3]:
import mlflow.pyfunc

NUM_FEATURES = ['Store', 'Dept', 'Temperature', 'Fuel_Price_Delta', 'CPI_Delta',
                'Unemployment_Delta', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4',
                'MarkDown5', 'Size', 'WeekOfYear', 'Is_Black_Friday', 'Pre_Christmas', 'IsHoliday']
FEAT_COLS = NUM_FEATURES + ['Type']


class TreeEnsembleModel(mlflow.pyfunc.PythonModel):
    def __init__(self, w_xgb):
        self.w_xgb = w_xgb

    def load_context(self, context):
        import joblib, pandas as pd
        self.xgb = joblib.load(context.artifacts['xgb'])
        self.lgb = joblib.load(context.artifacts['lgb'])
        self.train_raw = pd.read_parquet(context.artifacts['train_raw'])
        self.stores = pd.read_parquet(context.artifacts['stores'])
        self.features = pd.read_parquet(context.artifacts['features'])

    def _merge(self, raw):
        import pandas as pd
        df = raw.merge(self.stores, on='Store', how='left')
        df = df.merge(self.features, on=['Store', 'Date', 'IsHoliday'], how='left')
        df['Date'] = pd.to_datetime(df['Date'])
        return df

    def _engineer(self, df):
        df = df.copy()
        df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
        df['Year'] = df['Date'].dt.year
        df['Is_Black_Friday'] = ((df['Date'].dt.month == 11) & (df['Date'].dt.day >= 23) & (df['Date'].dt.day <= 29)).astype(int)
        df['Pre_Christmas'] = (df['WeekOfYear'] == 51).astype(int)
        df['IsHoliday'] = df['IsHoliday'].astype(int)
        df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
        for col in ['Fuel_Price', 'CPI', 'Unemployment']:
            df[col + '_Delta'] = df.groupby(['Store', 'Dept'])[col].diff().fillna(0)
        return df

    def predict(self, context, model_input):
        import numpy as np, pandas as pd
        test = model_input[['Store', 'Dept', 'Date', 'IsHoliday']].copy()
        te = self._merge(test.assign(Weekly_Sales=np.nan))
        tr = self._merge(self.train_raw)
        combined = self._engineer(pd.concat([tr, te], ignore_index=True))
        rows = combined[combined['Weekly_Sales'].isna()].copy()
        X = rows[FEAT_COLS]
        xp = np.clip(self.xgb.predict(X), 0, None)
        lp = np.clip(self.lgb.predict(X), 0, None)
        rows['pred'] = self.w_xgb * xp + (1 - self.w_xgb) * lp
        rows['key'] = rows['Store'].astype(str) + '_' + rows['Dept'].astype(str) + '_' + rows['Date'].dt.strftime('%Y-%m-%d')
        mi = model_input.copy()
        mi['key'] = mi['Store'].astype(str) + '_' + mi['Dept'].astype(str) + '_' + pd.to_datetime(mi['Date']).dt.strftime('%Y-%m-%d')
        out = mi.merge(rows[['key', 'pred']].drop_duplicates('key'), on='key', how='left')
        return out['pred'].fillna(0).clip(lower=0).values

D:\Desktop\ML Project\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


## Bundle artifacts, log + register

In [4]:
import joblib
import mlflow
import dagshub

dagshub.init(repo_owner='slosa23', repo_name='Walmart-Sales-Forecasting', mlflow=True)
mlflow.set_experiment('Champion_Registry')

os.makedirs('champion_artifacts', exist_ok=True)
joblib.dump(xgb_final, 'champion_artifacts/xgb_tree.joblib')
joblib.dump(lgb_final, 'champion_artifacts/lgb_tree.joblib')
train_raw.to_parquet('champion_artifacts/tree_train_raw.parquet')
stores.to_parquet('champion_artifacts/tree_stores.parquet')
features.to_parquet('champion_artifacts/tree_features.parquet')

with mlflow.start_run(run_name='Register_Tree_Ensemble'):
    mlflow.log_param('w_xgb', W_XGB)
    mlflow.log_param('w_lgb', round(1 - W_XGB, 3))
    mlflow.log_metric('val_WMAE', 1932.17)
    info = mlflow.pyfunc.log_model(
        name='model',
        python_model=TreeEnsembleModel(W_XGB),
        artifacts={
            'xgb': 'champion_artifacts/xgb_tree.joblib',
            'lgb': 'champion_artifacts/lgb_tree.joblib',
            'train_raw': 'champion_artifacts/tree_train_raw.parquet',
            'stores': 'champion_artifacts/tree_stores.parquet',
            'features': 'champion_artifacts/tree_features.parquet',
        },
        pip_requirements=['scikit-learn', 'xgboost', 'lightgbm', 'pandas', 'numpy', 'pyarrow'],
        registered_model_name='Walmart_Tree_Ensemble',
    )
    print('Registered:', info.model_uri)

Accessing as GigiSichinava

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

2026/07/12 22:18:41 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
Successfully registered model 'Walmart_Tree_Ensemble'.
2026/07/12 22:24:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_Tree_Ensemble, version 1
Created version '1' of model 'Walmart_Tree_Ensemble'.


Registered: models:/m-98d4fff8753c4a7e90a338a40b9a2908
🏃 View run Register_Tree_Ensemble at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/11/runs/2ff693025ac44cb59b80d92f91e31a85
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/11


## Verify: load it back from the registry and predict on raw test.csv

In [5]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
versions = client.search_model_versions("name='Walmart_Tree_Ensemble'")
latest = max(int(v.version) for v in versions)
print('Loading Walmart_Tree_Ensemble version', latest)
loaded = mlflow.pyfunc.load_model(f'models:/Walmart_Tree_Ensemble/{latest}')

test_raw = pd.read_csv('data/test.csv')
preds = loaded.predict(test_raw[['Store', 'Dept', 'Date', 'IsHoliday']])
submission = pd.DataFrame({
    'Id': test_raw['Store'].astype(str) + '_' + test_raw['Dept'].astype(str) + '_' + test_raw['Date'].astype(str),
    'Weekly_Sales': np.asarray(preds, dtype=float),
})
submission.to_csv('submission_tree_ensemble_registry.csv', index=False)
print('rows:', len(submission),
      '| NaNs:', int(submission['Weekly_Sales'].isna().sum()),
      '| negatives:', int((submission['Weekly_Sales'] < 0).sum()))
submission.head()

Loading Walmart_Tree_Ensemble version 1


rows: 115064 | NaNs: 0 | negatives: 0


,Id,Weekly_Sales
0,1_1_2012-11-02,35883.270411
1,1_1_2012-11-09,18963.630744
2,1_1_2012-11-16,20222.889820
3,1_1_2012-11-23,19534.432867
4,1_1_2012-11-30,24364.005326


## Notes

- `Walmart_Tree_Ensemble` loads from the registry and runs on the raw `test.csv`, like
  the TimesFM champion.
- The pyfunc rebuilds the test features on a combined train+test frame, so the delta
  features stay continuous across the train->test boundary.
- TimesFM (`Walmart_Champion_TimesFM`) remains the registered overall champion.